## Converting images to something that can be fed to a network 
- What format do networks expect exactly and how to find that info ?
- What if images all have different sizes to start with ? 
- Should we just always resize images to be squares ?

## Deep dive into the training loop : intuition 
Let's simulate an example with 3 batches of 3 images each. Each image is 128 by 128 pixels each. 



## Deep dive into the training loop : code

In the following case we are only changing weights and biases (aka training) for the last layer of the network here, since we use a pre trained network and only change the last layer to adapt it to our problem of binary classification of hot dogs vs no hot dogs --> model.fc : last step of network

Let's go through steps by steps :

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

#Cross Entropy Loss : measures how wrong predictions are
# combines softmax + negative log likelihood in one step
# 1. Softmax: [2.1, -0.5] → [0.93, 0.07]  (probabilities sum to 1)
# 2. If true label is 0 (hotdog), loss = -log(0.93) = 0.07 (low loss, correct!)
# 3. If true label is 1 (not hotdog), loss = -log(0.07) = 2.66 (high loss, wrong!)

# Optimizer : decides HOW to update weights and biases based on computed gradients

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)


# epoch : one complete pass through the entire training dataset
# one epoch = one forward pass + one backward pass of all the training examples (all batches)
num_epochs = 5
train_losses = []
train_accuracies = []

#outer loop iterates over epochs, inner loop iterates over batches of data
# why making batches ? 
# 1. Memory efficiency: processing the entire dataset at once may not fit in memory, especially for large datasets.
# 2. Faster convergence: updating weights after each batch can lead to faster convergence compared to updating after the entire dataset.
# GPUs can process batches in parallel, making training faster --> we don't use a GPU here, but in practice, using a GPU with batches can significantly speed up training.

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        # Move data to device : load data in CPU (since we don't use GPU here)
        images, labels = images.to(device), labels.to(device)
        
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Track statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    print(f'Epoch {epoch+1}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.2f}%')